# 🚀 YOLOv8n Training Pipeline for M1 MacBook

Train YOLOv8n on AR Electrical Panels dataset and export to CoreML

**Optimized for Apple M1 with MPS GPU acceleration** 🍎

## Step 1: Install Required Libraries

In [ ]:
!pip install ultralytics roboflow

## Step 2: Download Dataset from Roboflow

⚠️ **Replace YOUR_API_KEY with your actual Roboflow API key**

Get it from: https://app.roboflow.com/settings/api

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("mohameds-workspace-qsg6d").project("ar-electrical-panels")
version = project.version(1)
dataset = version.download("yolov8")

print(f"✅ Dataset downloaded to: {dataset.location}")

## Step 3: Train YOLOv8n Model with M1 GPU

Using MPS (Metal Performance Shaders) for M1 GPU acceleration

⏱️ Estimated time: 15-30 minutes on M1

In [ ]:
from ultralytics import YOLO
import torch

# Check M1 GPU availability
if torch.backends.mps.is_available():
    device = 'mps'
    print("✅ Using MPS (M1 GPU Acceleration)")
else:
    device = 'cpu'
    print("⚠️ Using CPU (MPS not available)")

# Load YOLOv8n model
model = YOLO('yolov8n.pt')

# Train the model
print("\n🚀 Starting training...")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    device=device,
    project='runs/detect',
    name='electrical_panels',
    patience=10,
    save=True,
    plots=True
)

print("\n✅ Training completed!")

## Step 4: View Training Metrics (Loss Graphs & mAP)

In [ ]:
from IPython.display import Image, display
import os

results_path = 'runs/detect/electrical_panels'

print("📊 Training Results:\n")

# Loss curves and mAP scores
if os.path.exists(f'{results_path}/results.png'):
    print("Loss Curves & mAP Scores:")
    display(Image(filename=f'{results_path}/results.png'))

# Confusion matrix
if os.path.exists(f'{results_path}/confusion_matrix.png'):
    print("\nConfusion Matrix:")
    display(Image(filename=f'{results_path}/confusion_matrix.png'))

# Validation predictions
if os.path.exists(f'{results_path}/val_batch0_pred.jpg'):
    print("\nValidation Predictions:")
    display(Image(filename=f'{results_path}/val_batch0_pred.jpg'))

## Step 5: Export to CoreML for iPhone/MacBook

In [ ]:
# Load best trained model
best_model = YOLO('runs/detect/electrical_panels/weights/best.pt')

print("🔄 Exporting to CoreML...")

# Export to CoreML
coreml_model = best_model.export(
    format='coreml',
    imgsz=640,
    nms=True,
    half=False
)

print(f"\n✅ CoreML model exported!")
print(f"📁 Location: {coreml_model}")

## Step 6: Test Model Predictions

In [ ]:
import glob

# Find test images
test_images = glob.glob(f'{dataset.location}/test/images/*')

if test_images:
    test_image = test_images[0]
    print(f"Testing on: {test_image}")
    
    # Run inference
    results = best_model.predict(
        source=test_image,
        conf=0.5,
        save=True,
        show_labels=True,
        show_conf=True
    )
    
    # Display results
    pred_path = results[0].save_dir
    pred_images = glob.glob(f'{pred_path}/*.jpg') + glob.glob(f'{pred_path}/*.png')
    
    if pred_images:
        print("\n🎯 Prediction Results:")
        display(Image(filename=pred_images[0]))
    
    print(f"\n✅ Results saved to: {pred_path}")
else:
    print("⚠️ No test images found")

## 🎉 Pipeline Complete!

Your CoreML model is ready at:
```
runs/detect/electrical_panels/weights/best.mlpackage
```

Use this file in your iOS/macOS app! 📱💻

In [ ]:
# Final summary
print("=" * 60)
print("🎉 TRAINING PIPELINE COMPLETE!")
print("=" * 60)
print(f"\n📁 Model Files:")
print(f"   PyTorch: runs/detect/electrical_panels/weights/best.pt")
print(f"   CoreML:  runs/detect/electrical_panels/weights/best.mlpackage")
print(f"\n📊 Metrics: runs/detect/electrical_panels/results.png")
print(f"\n✅ Ready for deployment!")